<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/ResNet_SRCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [100]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [136]:
import os
import cv2
import h5py
import numpy as np
from tqdm import tqdm

# Parametreler
TRAINING_DATA_PATH = "/content/drive/MyDrive/srcnn_dataset/dataset/train"
TEST_DATA_PATH = "/content/drive/MyDrive/srcnn_dataset/dataset/test"
RANDOM_CROP_COUNT = 30
INPUT_PATCH_SIZE = 32
OUTPUT_PATCH_SIZE = 20
CONV_BORDER = 6
SCALE_FACTOR = 2
BLOCK_SIZE = 32
BLOCK_STEP = 16

"""
def process_image(image_path):
    ""
    Y kanalını al ve LR/HR çiftini oluştur.
    ""
    hr_image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    hr_image = cv2.cvtColor(hr_image, cv2.COLOR_BGR2YCrCb)[:, :, 0]
    height, width = hr_image.shape

    # Low-resolution oluştur (bicubic downsampling)
    lr_image = cv2.resize(hr_image, (width // SCALE_FACTOR, height // SCALE_FACTOR))
    lr_image = cv2.resize(lr_image, (width, height))

    return lr_image, hr_image
"""

def process_image(image_path):
    """
    Y kanalını al ve LR/HR çiftini oluştur.
    """
    hr_image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    hr_image = cv2.cvtColor(hr_image, cv2.COLOR_BGR2YCrCb)[:, :, 0]
    height, width = hr_image.shape

    # Low-resolution oluştur (bicubic downsampling)
    lr_image = cv2.resize(hr_image, (width // SCALE_FACTOR, height // SCALE_FACTOR), interpolation=cv2.INTER_CUBIC)
    lr_image = cv2.resize(lr_image, (width, height), interpolation=cv2.INTER_CUBIC)

    # Padding: Kenar artefaktlarını engellemek için reflect padding
    lr_image = cv2.copyMakeBorder(lr_image, 6, 6, 6, 6, cv2.BORDER_REFLECT)
    hr_image = cv2.copyMakeBorder(hr_image, 6, 6, 6, 6, cv2.BORDER_REFLECT)

    return lr_image, hr_image
"""
def generate_patches(lr, hr, crop_count=RANDOM_CROP_COUNT):
    ""
    Rastgele kırpılmış patch'ler oluştur.
    ""
    h, w = hr.shape
    max_pos = min(h, w) - INPUT_PATCH_SIZE

    for _ in range(crop_count):
        x = np.random.randint(0, max_pos)
        y = np.random.randint(0, max_pos)

        lr_patch = lr[x:x + INPUT_PATCH_SIZE, y:y + INPUT_PATCH_SIZE]
        hr_patch = hr[x:x + INPUT_PATCH_SIZE, y:y + INPUT_PATCH_SIZE]

        # Kenar kırpma
        hr_patch = hr_patch[CONV_BORDER:-CONV_BORDER, CONV_BORDER:-CONV_BORDER]

        yield lr_patch / 255.0, hr_patch / 255.0
"""

def generate_patches(lr, hr, crop_count=RANDOM_CROP_COUNT):
    """
    Rastgele kırpılmış patch'ler oluştur.
    """
    h, w = hr.shape
    max_pos = min(h, w) - INPUT_PATCH_SIZE

    for _ in range(crop_count):
        x = np.random.randint(0, max_pos)
        y = np.random.randint(0, max_pos)

        lr_patch = lr[x:x + INPUT_PATCH_SIZE, y:y + INPUT_PATCH_SIZE]
        hr_patch = hr[x:x + INPUT_PATCH_SIZE, y:y + INPUT_PATCH_SIZE]

        # Kenar kırpma işlemini kaldırıyoruz!
        yield lr_patch / 255.0, hr_patch / 255.0

def write_to_hdf5(h5_file, image_folder, crop_count=RANDOM_CROP_COUNT):
    """
    Veriyi doğrudan HDF5'e batch halinde yaz.
    """
    image_files = sorted(os.listdir(image_folder))

    for filename in tqdm(image_files):
        image_path = os.path.join(image_folder, filename)
        lr, hr = process_image(image_path)

        for lr_patch, hr_patch in generate_patches(lr, hr, crop_count):
            h5_file['data'].resize((h5_file['data'].shape[0] + 1), axis=0)
            h5_file['label'].resize((h5_file['label'].shape[0] + 1), axis=0)
            h5_file['data'][-1] = lr_patch[np.newaxis, :, :]

            # Resize hr_patch before assigning to 'label' dataset
            hr_patch_resized = cv2.resize(hr_patch, (OUTPUT_PATCH_SIZE, OUTPUT_PATCH_SIZE), interpolation=cv2.INTER_CUBIC)
            h5_file['label'][-1] = hr_patch_resized[np.newaxis, :, :] # Assign the resized patch

def create_h5_file(output_path):
    """
    HDF5 dosyası oluştur.
    """
    with h5py.File(output_path, 'w') as h5_file:
        h5_file.create_dataset('data', shape=(0, 1, INPUT_PATCH_SIZE, INPUT_PATCH_SIZE), maxshape=(None, 1, INPUT_PATCH_SIZE, INPUT_PATCH_SIZE), dtype=np.float32)
        h5_file.create_dataset('label', shape=(0, 1, OUTPUT_PATCH_SIZE, OUTPUT_PATCH_SIZE), maxshape=(None, 1, OUTPUT_PATCH_SIZE, OUTPUT_PATCH_SIZE), dtype=np.float32)

        return h5_file.filename

if __name__ == "__main__":
    # Eğitim verisi
    train_h5_path = create_h5_file("/content/drive/MyDrive/srcnn_dataset/train.h5")
    with h5py.File(train_h5_path, 'a') as train_h5:
        write_to_hdf5(train_h5, TRAINING_DATA_PATH)

    # Test verisi
    test_h5_path = create_h5_file("/content/drive/MyDrive/srcnn_dataset/test.h5")
    with h5py.File(test_h5_path, 'a') as test_h5:
        write_to_hdf5(test_h5, TEST_DATA_PATH)

    print("✅ HDF5 Dataset Oluşturuldu!")


100%|██████████| 640/640 [01:16<00:00,  8.35it/s]

✅ HDF5 Dataset Oluşturuldu!


In [102]:
import os
import numpy as np
import math
import cv2
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Add, LeakyReLU, Activation, Lambda
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from tensorflow import keras
from IPython import get_ipython
from IPython.display import display

In [137]:
from tensorflow.keras.layers import Input, Conv2D, Add, Lambda, ReLU
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

# PSNR Metriği
def psnr(y_true, y_pred):
    return tf.image.psnr(y_true, y_pred, max_val=255.0)

def res_block(x, filters=64, kernel_size=(3, 3)):
    skip = x
    x = Conv2D(filters, kernel_size, padding='same', activation='relu')(x)
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = Add()([x, skip])  # Residual bağlantı
    return x

def create_resnet_srcnn():
    inputs = Input(shape=(32, 32, 1))

    # Normalizasyon
    x = Lambda(lambda x: x / 255.0)(inputs)

    # İlk Feature Extraction Katmanı
    x = Conv2D(64, (9, 9), padding='same', activation='relu')(x)

    # ResNet Blokları (SRCNN için derinleştirilmiş yapı)
    for _ in range(5):
        x = res_block(x)

    # Non-linear Mapping
    x = Conv2D(32, (5, 5), padding='same', activation='relu')(x)

    # Reconstruction Katmanı
    x = Conv2D(1, (5, 5), padding='same')(x)

    # Orijinal piksel aralığına dönüş
    outputs = Lambda(lambda x: x * 255.0)(x)

    model = Model(inputs=inputs, outputs=outputs)

    # L1 Loss (MAE) + PSNR metrik
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mae', metrics=[psnr])

    return model


In [103]:
"""
def create_training_model():
    ""
    ResNet mimarisi tabanlı SRCNN (Super-Resolution Convolutional Neural Network) modeli.
    32x32x1 boyutundaki giriş görüntülerini işler ve süper çözünürlük üretir.
    Derin residual bloklar kullanılarak gradyan kaybolması problemi çözülür.
    PSNR performansı optimize edilmiştir.
    ""

    # Giriş katmanı
    inputs = Input(shape=(32, 32, 1))

    # Girişi normalize et (0-1 aralığı)
    x = Lambda(lambda x: x / 255.0)(inputs)

    # İlk özellik çıkarımı katmanı
    x = Conv2D(64, kernel_size=3, padding='same')(x)
    x = Activation('relu')(x)

    # İlk özellik haritasını sakla - global residual learning için
    initial_features = x

    # ResNet blokları
    def residual_block(x, filters=64, kernel_size=3):
        """Standart ResNet bloğu"""
        # Shortcut bağlantısı
        shortcut = x

        # İlk konvolüsyon katmanı
        x = Conv2D(filters, kernel_size=kernel_size, padding='same')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)

        # İkinci konvolüsyon katmanı
        x = Conv2D(filters, kernel_size=kernel_size, padding='same')(x)
        x = BatchNormalization()(x)

        # Shortcut bağlantısı ekle
        x = Add()([x, shortcut])
        x = Activation('relu')(x)

        return x

    # ResNet blokları ekle
    num_residual_blocks = 16  # Derin bir model için 16 blok kullanıyoruz
    for _ in range(num_residual_blocks):
        x = residual_block(x)

    # Özellik haritalarını birleştirme
    x = Conv2D(64, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, initial_features])  # Global residual learning

    # Rekonstrüksiyon katmanları
    x = Conv2D(64, kernel_size=3, padding='same')(x)
    x = Activation('relu')(x)

    x = Conv2D(32, kernel_size=3, padding='same')(x)
    x = Activation('relu')(x)

    # Çıkış katmanı
    x = Conv2D(1, kernel_size=3, padding='same')(x)

    # Çıkışı orijinal değer aralığına yeniden ölçeklendir
    x = Lambda(lambda x: x * 255.0)(x)

    # Son çıkış - orijinal görüntü ile artık bağlantı
    outputs = Add()([x, inputs])

    # Modeli oluştur
    model = Model(inputs=inputs, outputs=outputs)

    # PSNR metriği tanımla
    def psnr(y_true, y_pred):
        return tf.image.psnr(y_true, y_pred, max_val=255.0)

    # SSIM metriği tanımla
    def ssim(y_true, y_pred):
        return tf.image.ssim(y_true, y_pred, max_val=255.0)

    # Öğrenme oranı scheduler'ı
    initial_learning_rate = 0.001
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=2000,
        decay_rate=0.9,
        staircase=True)

    # Model derleme
    optimizer = Adam(learning_rate=lr_schedule)
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=[psnr, ssim]
    )

    return model
"""

In [ ]:
"""
def create_training_model():
    ""
    Eğitim için geliştirilmiş ResNet tabanlı SRCNN modelini oluşturur.
    Sabit boyutlu giriş şekli (32x32x1) kullanır.
    PSNR performansını artırmak için optimize edilmiştir.
    Residual bloklar ile derin öğrenme kapasitesi artırılmıştır.
    ""
    # Giriş katmanını tanımla
    inputs = Input(shape=(32, 32, 1))

    # İlk özellik çıkarma katmanı
    x = Conv2D(
        filters=64,
        kernel_size=(7, 7),
        kernel_initializer='he_normal',
        padding='same',
        use_bias=False
    )(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Residual bloklar
    def residual_block(x, filters, kernel_size=3):
        identity = x

        x = Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            kernel_initializer='he_normal',
            use_bias=False
        )(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)

        x = Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            kernel_initializer='he_normal',
            use_bias=False
        )(x)
        x = BatchNormalization()(x)

        x = Add()([x, identity])
        x = Activation('relu')(x)
        return x

    # Residual blokları ekle
    x = residual_block(x, filters=64)
    x = residual_block(x, filters=64)

    # Global artık öğrenme
    global_res = x

    # Yeniden yapılandırma katmanları
    x = Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=False
    )(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Son katman
    x = Conv2D(
        filters=1,
        kernel_size=(3, 3),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=True
    )(x)

    # Global artık bağlantıyı ekle
    outputs = Add()([x, inputs])

    # Modeli oluştur
    model = Model(inputs=inputs, outputs=outputs)

    # Öğrenme oranı scheduler'ı
    initial_learning_rate = 0.001
    decay_steps = 1000
    decay_rate = 0.9
    learning_rate_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=decay_steps,
        decay_rate=decay_rate,
        staircase=True
    )

    # Model derleme
    adam_optimizer = Adam(learning_rate=learning_rate_schedule)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error']
    )

    return model
"""

In [138]:
def res_block(x, filters=64, kernel_size=(3, 3)):
    skip = x
    x = Conv2D(filters, kernel_size, padding='same', activation='relu')(x)
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = Add()([x, skip])  # Residual bağlantı
    return x

def prediction_resnet_srcnn():
    inputs = Input(shape=(None, None, 1))

    # Normalizasyon
    x = Lambda(lambda x: x / 255.0)(inputs)

    # İlk Feature Extraction Katmanı
    x = Conv2D(64, (9, 9), padding='same', activation='relu')(x)

    # ResNet Blokları (SRCNN için derinleştirilmiş yapı)
    for _ in range(5):
        x = res_block(x)

    # Non-linear Mapping
    x = Conv2D(32, (5, 5), padding='same', activation='relu')(x)

    # Reconstruction Katmanı
    x = Conv2D(1, (5, 5), padding='same')(x)

    # Orijinal piksel aralığına dönüş
    outputs = Lambda(lambda x: x * 255.0)(x)

    model = Model(inputs=inputs, outputs=outputs)

    # L1 Loss (MAE) + PSNR metrik
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mae', metrics=[psnr])

    return model

In [104]:
"""
def create_prediction_model():
    ""
    ResNet mimarisi tabanlı SRCNN (Super-Resolution Convolutional Neural Network) modeli.
    32x32x1 boyutundaki giriş görüntülerini işler ve süper çözünürlük üretir.
    Derin residual bloklar kullanılarak gradyan kaybolması problemi çözülür.
    PSNR performansı optimize edilmiştir.
    ""

    # Giriş katmanı
    inputs = Input(shape=(None, None, 1))

    # Girişi normalize et (0-1 aralığı)
    x = Lambda(lambda x: x / 255.0)(inputs)

    # İlk özellik çıkarımı katmanı
    x = Conv2D(64, kernel_size=3, padding='same')(x)
    x = Activation('relu')(x)

    # İlk özellik haritasını sakla - global residual learning için
    initial_features = x

    # ResNet blokları
    def residual_block(x, filters=64, kernel_size=3):
        ""Standart ResNet bloğu""
        # Shortcut bağlantısı
        shortcut = x

        # İlk konvolüsyon katmanı
        x = Conv2D(filters, kernel_size=kernel_size, padding='same')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)

        # İkinci konvolüsyon katmanı
        x = Conv2D(filters, kernel_size=kernel_size, padding='same')(x)
        x = BatchNormalization()(x)

        # Shortcut bağlantısı ekle
        x = Add()([x, shortcut])
        x = Activation('relu')(x)

        return x

    # ResNet blokları ekle
    num_residual_blocks = 16  # Derin bir model için 16 blok kullanıyoruz
    for _ in range(num_residual_blocks):
        x = residual_block(x)

    # Özellik haritalarını birleştirme
    x = Conv2D(64, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, initial_features])  # Global residual learning

    # Rekonstrüksiyon katmanları
    x = Conv2D(64, kernel_size=3, padding='same')(x)
    x = Activation('relu')(x)

    x = Conv2D(32, kernel_size=3, padding='same')(x)
    x = Activation('relu')(x)

    # Çıkış katmanı
    x = Conv2D(1, kernel_size=3, padding='same')(x)

    # Çıkışı orijinal değer aralığına yeniden ölçeklendir
    x = Lambda(lambda x: x * 255.0)(x)

    # Son çıkış - orijinal görüntü ile artık bağlantı
    outputs = Add()([x, inputs])

    # Modeli oluştur
    model = Model(inputs=inputs, outputs=outputs)

    # PSNR metriği tanımla
    def psnr(y_true, y_pred):
        return tf.image.psnr(y_true, y_pred, max_val=255.0)

    # SSIM metriği tanımla
    def ssim(y_true, y_pred):
        return tf.image.ssim(y_true, y_pred, max_val=255.0)

    # Öğrenme oranı scheduler'ı
    initial_learning_rate = 0.001
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=2000,
        decay_rate=0.9,
        staircase=True)

    # Model derleme
    optimizer = Adam(learning_rate=lr_schedule)
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=[psnr, ssim]
    )

    return model
"""

In [ ]:
"""
def prediction_model():
    # Giriş katmanını tanımla
    inputs = Input(shape=(32, 32, 1))

    # İlk özellik çıkarma katmanı
    x = Conv2D(
        filters=64,
        kernel_size=(7, 7),
        kernel_initializer='he_normal',
        padding='same',
        use_bias=False
    )(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Residual bloklar
    def residual_block(x, filters, kernel_size=3):
        identity = x

        x = Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            kernel_initializer='he_normal',
            use_bias=False
        )(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)

        x = Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            kernel_initializer='he_normal',
            use_bias=False
        )(x)
        x = BatchNormalization()(x)

        x = Add()([x, identity])
        x = Activation('relu')(x)
        return x

    # Residual blokları ekle
    x = residual_block(x, filters=64)
    x = residual_block(x, filters=64)

    # Global artık öğrenme
    global_res = x

    # Yeniden yapılandırma katmanları
    x = Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=False
    )(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Son katman
    x = Conv2D(
        filters=1,
        kernel_size=(3, 3),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=True
    )(x)

    # Global artık bağlantıyı ekle
    outputs = Add()([x, inputs])

    # Modeli oluştur
    model = Model(inputs=inputs, outputs=outputs)

    # Öğrenme oranı scheduler'ı
    initial_learning_rate = 0.001
    decay_steps = 1000
    decay_rate = 0.9
    learning_rate_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=decay_steps,
        decay_rate=decay_rate,
        staircase=True
    )

    # Model derleme
    adam_optimizer = Adam(learning_rate=learning_rate_schedule)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error']
    )

    return model
"""

In [139]:
def train_model():
    """
    SRCNN modelini eğitir ve en iyi modeli kaydeder.
    """
    # Eğitim modelini oluştur
    srcnn_model = create_resnet_srcnn()
    print(srcnn_model.summary())

    # Eğitim ve doğrulama verilerini yükle
    print("Eğitim verilerini yükleme...")
    train_data, train_labels = load_h5_data("/content/drive/MyDrive/srcnn_dataset/train.h5")
    val_data, val_labels = load_h5_data("/content/drive/MyDrive/srcnn_dataset/test.h5")

    # Reshape the data
    train_data = np.transpose(train_data, (0, 2, 3, 1))  # Change to (None, 32, 32, 1)
    val_data = np.transpose(val_data, (0, 2, 3, 1))    # Change to (None, 32, 32, 1)
    train_labels = np.transpose(train_labels,(0,2,3,1))
    val_labels = np.transpose(val_labels,(0,2,3,1))

    # Model çıkışıyla eşleşmesi için etiketleri yeniden boyutlandırın
    # Aşağıdaki satırı yorumdan çıkarın ve etiketleri 32x32 olarak yeniden boyutlandırın
    import tensorflow as tf
    train_labels = tf.image.resize(train_labels, [32, 32], method='bicubic')
    val_labels = tf.image.resize(val_labels, [32, 32], method='bicubic')


    # Model kaydetme için callback oluştur
    checkpoint = ModelCheckpoint(
        "ResNet_SRCNN2.h5",
        monitor='val_loss',
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode='min'
    )
    callbacks_list = [checkpoint]

    # Modeli eğit
    print("Model eğitimi başlıyor...")
    srcnn_model.fit(
        train_data, train_labels,
        batch_size=16,
        validation_data=(val_data, val_labels),
        callbacks=callbacks_list,
        shuffle=True,
        epochs=100,
        verbose=1
    )

    print("Eğitim tamamlandı!")

In [140]:
if __name__ == "__main__":
    train_model()

Model: "functional_40"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_28            │ (None, 32, 32, 1)      │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lambda_12 (Lambda)        │ (None, 32, 32, 1)      │              0 │ input_layer_28[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_327 (Conv2D)       │ (None, 32, 32, 64)     │          5,248 │ lambda_12[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_328 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_327[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_329 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_328[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_138 (Add)             │ (None, 32, 32, 64)     │              0 │ conv2d_329[0][0],      │
│                           │                        │                │ conv2d_327[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_330 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_138[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_331 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_330[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_139 (Add)             │ (None, 32, 32, 64)     │              0 │ conv2d_331[0][0],      │
│                           │                        │                │ add_138[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_332 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_139[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_333 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_332[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_140 (Add)             │ (None, 32, 32, 64)     │              0 │ conv2d_333[0][0],      │
│                           │                        │                │ add_139[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_334 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_140[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_335 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_334[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_141 (Add)             │ (None, 32, 32, 64)     │              0 │ conv2d_335[0][0],      │
│                           │                        │                │ add_140[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_336 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_141[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_337 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_336[0][0]       │
├──────────────────────

 Total params: 426,561 (1.63 MB)

 Trainable params: 426,561 (1.63 MB)

 Non-trainable params: 0 (0.00 B)

...

4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0201 - psnr: 81.0184 - val_loss: 0.0102 - val_psnr: 88.2443
Epoch 61/100
4165/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0201 - psnr: 81.0620
Epoch 61: val_loss did not improve from 0.01024
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0201 - psnr: 81.0620 - val_loss: 0.0108 - val_psnr: 87.1300
Epoch 62/100
4168/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0201 - psnr: 81.0658
Epoch 62: val_loss did not improve from 0.01024
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0201 - psnr: 81.0657 - val_loss: 0.0111 - val_psnr: 86.7648
Epoch 63/100
4164/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0199 - psnr: 81.1469
Epoch 63: val_loss did not improve from 0.01024
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0199 - psnr: 81.1468 - val_loss: 0.0104 - val_psnr: 87.7977
Epoch 64/100
4164/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0200 - psnr: 81.1062
Epoch 64: val_loss did not improve from 0.01024
4172

4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0200 - psnr: 81.1143 - val_loss: 0.0101 - val_psnr: 88.2162
Epoch 69/100
4163/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0200 - psnr: 81.1461
Epoch 69: val_loss did not improve from 0.01015
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0200 - psnr: 81.1460 - val_loss: 0.0104 - val_psnr: 87.8762
Epoch 70/100
4168/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0200 - psnr: 81.1104
Epoch 70: val_loss did not improve from 0.01015
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0200 - psnr: 81.1105 - val_loss: 0.0111 - val_psnr: 86.9304
Epoch 71/100
4165/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0200 - psnr: 81.1221
Epoch 71: val_loss did not improve from 0.01015
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0200 - psnr: 81.1221 - val_loss: 0.0105 - val_psnr: 87.7771
Epoch 72/100
4166/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0199 - psnr: 81.1739
Epoch 72: val_loss did not improve from 0.01015
4172

4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0199 - psnr: 81.1875 - val_loss: 0.0101 - val_psnr: 88.3485
Epoch 81/100
4166/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0197 - psnr: 81.3084
Epoch 81: val_loss did not improve from 0.01007
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0197 - psnr: 81.3083 - val_loss: 0.0104 - val_psnr: 87.8406
Epoch 82/100
4167/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0198 - psnr: 81.2376
Epoch 82: val_loss did not improve from 0.01007
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0198 - psnr: 81.2376 - val_loss: 0.0116 - val_psnr: 86.3871
Epoch 83/100
4165/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0198 - psnr: 81.2553
Epoch 83: val_loss did not improve from 0.01007
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0198 - psnr: 81.2552 - val_loss: 0.0102 - val_psnr: 88.1909
Epoch 84/100
4169/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0199 - psnr: 81.1643
Epoch 84: val_loss did not improve from 0.01007
4172

4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0198 - psnr: 81.2719 - val_loss: 0.0100 - val_psnr: 88.3084
Epoch 88/100
4167/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0200 - psnr: 81.1224
Epoch 88: val_loss did not improve from 0.01002
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0200 - psnr: 81.1224 - val_loss: 0.0102 - val_psnr: 88.2332
Epoch 89/100
4167/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0196 - psnr: 81.3235
Epoch 89: val_loss did not improve from 0.01002
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0196 - psnr: 81.3234 - val_loss: 0.0108 - val_psnr: 87.2579
Epoch 90/100
4166/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0197 - psnr: 81.2577
Epoch 90: val_loss did not improve from 0.01002
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0197 - psnr: 81.2575 - val_loss: 0.0175 - val_psnr: 82.3327
Epoch 91/100
4167/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0205 - psnr: 80.8561
Epoch 91: val_loss did not improve from 0.01002
4172

4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0195 - psnr: 81.3985 - val_loss: 0.0099 - val_psnr: 88.5418
Epoch 98/100
4165/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0196 - psnr: 81.3625
Epoch 98: val_loss did not improve from 0.00992
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0196 - psnr: 81.3624 - val_loss: 0.0099 - val_psnr: 88.5628
Epoch 99/100
4163/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0197 - psnr: 81.3248
Epoch 99: val_loss did not improve from 0.00992
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0197 - psnr: 81.3248 - val_loss: 0.0101 - val_psnr: 88.1763
Epoch 100/100
4164/4172 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0196 - psnr: 81.3566
Epoch 100: val_loss did not improve from 0.00992
4172/4172 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - loss: 0.0196 - psnr: 81.3565 - val_loss: 0.0100 - val_psnr: 88.4677
Eğitim tamamlandı!


In [146]:
def predict_image(model_path, image_path, output_folder="/content/drive/MyDrive/srcnn_dataset/content/drive/MyDrive/srcnn_dataset/results"):
    """
    Bir görüntüyü SRCNN ile süper çözünürlüklü hale getirir.

        model_path: Eğitilmiş model ağırlıklarının yolu
        image_path: Girdi görüntüsünün yolu
        output_folder: Sonuçların kaydedileceği klasör
    """
    # Çıktı klasörünü oluştur
    os.makedirs(output_folder, exist_ok=True)

    # Dosya adı bilgilerini hazırla
    base_name = os.path.basename(image_path)
    file_name, _ = os.path.splitext(base_name)
    input_path = os.path.join(output_folder, f"{file_name}_bicubic.png")
    output_path = os.path.join(output_folder, f"{file_name}_srcnn.png")

    # Tahmin modelini oluştur ve ağırlıkları yükle
    srcnn_model = create_resnet_srcnn()
    srcnn_model.load_weights(model_path)
    print(f"Model yüklendi: {model_path}")

    # Orijinal görüntüyü yükle
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # BGR'dan YCrCb'ye dönüştür
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    height, width = img_ycrcb.shape[:2]

    # Y kanalını önce küçült sonra bicubic ile büyüt (düşük çözünürlük simulasyonu)
    y_channel = img_ycrcb[:, :, 0]
    y_channel_lr = cv2.resize(y_channel, (width // 2, height // 2), cv2.INTER_CUBIC)
    y_channel_bicubic = cv2.resize(y_channel_lr, (width, height), cv2.INTER_CUBIC)

    # Bicubic sonucunu kaydet
    img_bicubic = img_ycrcb.copy()
    img_bicubic[:, :, 0] = y_channel_bicubic
    img_bicubic = cv2.cvtColor(img_bicubic, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(input_path, img_bicubic)
    print(f"Bicubic upscaled görüntü kaydedildi: {input_path}")

    # SRCNN için girdiyi hazırla - Görüntüyü 32x32 patch'lere böl
    input_patches = []
    for y in range(0, height - 32, 32):
        for x in range(0, width - 32, 32):
            patch = y_channel_bicubic[y:y + 32, x:x + 32]
            input_patches.append(patch)

    input_data = np.array(input_patches).reshape(-1, 32, 32, 1) / 255.0

    # SRCNN tahmini yap
    predictions = srcnn_model.predict(input_data, batch_size=1) * 255.0

    # Tahminleri birleştirerek tam görüntüyü oluştur
    output_image = np.zeros_like(y_channel_bicubic, dtype=np.uint8)

    patch_index = 0
    for y in range(0, height - 32, 32):
        for x in range(0, width - 32, 32):
            resized_prediction = cv2.resize(predictions[patch_index, :, :, 0], (32, 32))
            output_image[y:y + 32, x:x + 32] = resized_prediction
            patch_index += 1

    # Değerleri [0, 255] aralığına kırp
    output_image = np.clip(output_image, 0, 255).astype(np.uint8)

    # Tahmini orijinal görüntüye yerleştir (evrişim padding nedeniyle 6 piksel kenarları kırpılır)
    img_srcnn = img_ycrcb.copy()

    img_srcnn[6:-6, 6:-6, 0] = output_image[6:-6, 6:-6]
    img_srcnn = cv2.cvtColor(img_srcnn, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(output_path, img_srcnn)
    print(f"SRCNN sonucu kaydedildi: {output_path}")

    # PSNR hesaplama
    original_y = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    bicubic_y = cv2.cvtColor(img_bicubic, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    srcnn_y = cv2.cvtColor(img_srcnn, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]

    bicubic_psnr = calculate_psnr(original_y, bicubic_y)
    srcnn_psnr = calculate_psnr(original_y, srcnn_y)

    print(f"Bicubic PSNR: {bicubic_psnr:.2f} dB")
    print(f"SRCNN PSNR: {srcnn_psnr:.2f} dB")
    print(f"PSNR İyileştirmesi: {srcnn_psnr - bicubic_psnr:.2f} dB")

In [149]:
predict_image(
        model_path="/content/ResNet_SRCNN2.h5",
        image_path="/content/drive/MyDrive/srcnn_dataset/dataset/train/M0601_img000287.jpg",
        output_folder="/content/drive/MyDrive/srcnn_dataset/results"
    )

Model yüklendi: /content/ResNet_SRCNN2.h5
Bicubic upscaled görüntü kaydedildi: /content/drive/MyDrive/srcnn_dataset/results/M0601_img000287_bicubic.png
496/496 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
SRCNN sonucu kaydedildi: /content/drive/MyDrive/srcnn_dataset/results/M0601_img000287_srcnn.png
Bicubic PSNR: 30.65 dB
SRCNN PSNR: 30.22 dB
PSNR İyileştirmesi: -0.43 dB
